# The 2026 NeuroGolf Championship

## Score: 5050.25

In [ ]:
from pathlib import Path
import hashlib
import zipfile
import re
import json
import os
import tempfile
import io
import contextlib
from collections import defaultdict, Counter

import numpy as np
import onnx

try:
    import onnxruntime as ort
except Exception:
    ort = None

try:
    import neurogolf_utils as nu
except Exception:
    nu = None

EXPECTED_FILE_COUNT = 401
EXPECTED_PUBLIC_SCORE = "6335.19"
EXPECTED_MANIFEST_SHA256 = "65e5d817b3195de894b8e1bb97045ed8ae6eeba3b4d79ab8dfcb4c4b88aaa13a"
OUTPUT_ZIP = Path("submission.zip")
MAX_ONNX_BYTES = int(1.44 * 1024 * 1024)
BANNED_OPS = {"LOOP", "SCAN", "NONZERO", "UNIQUE", "SCRIPT", "FUNCTION", "COMPRESS"}
MAX_VALID_CANDIDATES_PER_TASK = 512

input_root = Path("/kaggle/input")
preferred = input_root / "neurogolf-6335-19-controlled-public-artifact"
_seen_roots = set()
search_roots = []
def _add_search_root(p):
    p = Path(p)
    if not p.exists():
        return
    k = p.resolve()
    if k not in _seen_roots:
        _seen_roots.add(k)
        search_roots.append(p)
_add_search_root(preferred)
for _pat in ("*6335*artifact*", "*neurogolf*artifact*", "*blend*", "*neurogolf*"):
    for _p in sorted(input_root.glob(_pat)):
        _add_search_root(_p)
for _z in sorted(input_root.rglob("submission.zip")):
    _add_search_root(_z.parent)
for _hint in input_root.rglob("task001.onnx"):
    _add_search_root(_hint.parent)
use_full_validation = True


def normalize_task_name(name: str):
    m = re.search(r"task(\d{3})", name.lower())
    if not m:
        return None
    return f"task{m.group(1)}.onnx"


def source_rank(source_name: str):
    s = source_name.lower()
    if "6335" in s:
        return 0
    if "controlled" in s:
        return 1
    if "6285" in s or "6233" in s or "6254" in s or "konbu" in s:
        return 2
    if "blend" in s or "beicicc" in s or "afr1ste" in s:
        return 3
    if "synth:" in s:
        return 4
    return 9


def manifest_from_items(items):
    lines = []
    for name, data in sorted(items, key=lambda x: x[0]):
        lines.append(f"{name}\t{len(data)}\t{hashlib.sha256(data).hexdigest()}")
    return hashlib.sha256("\n".join(lines).encode()).hexdigest()


def load_zip_models(zip_path: Path):
    out = {}
    with zipfile.ZipFile(zip_path, "r") as zf:
        for name in zf.namelist():
            if not name.lower().endswith(".onnx"):
                continue
            tn = normalize_task_name(Path(name).name)
            if tn is None:
                continue
            out[tn] = zf.read(name)
    return out


def load_dir_models(root: Path):
    out = {}
    for p in root.rglob("task*.onnx"):
        tn = normalize_task_name(p.name)
        if tn is None:
            continue
        out[tn] = p.read_bytes()
    return out


def one_hot_grid(grid):
    t = np.zeros((1, 10, 30, 30), dtype=np.float32)
    for r, row in enumerate(grid):
        if r >= 30:
            break
        for c, v in enumerate(row):
            if c >= 30:
                break
            if 0 <= v < 10:
                t[0, v, r, c] = 1.0
    return t


def is_processable_model(raw: bytes):
    try:
        if len(raw) > MAX_ONNX_BYTES:
            return False
        model = onnx.load_model_from_string(raw)
        onnx.checker.check_model(model, full_check=True)
        for node in model.graph.node:
            if node.op_type.upper() in BANNED_OPS:
                return False
        inferred = onnx.shape_inference.infer_shapes(model, strict_mode=True)
        graph = inferred.graph
        for item in list(graph.input) + list(graph.value_info) + list(graph.output):
            if not item.type.HasField("tensor_type"):
                continue
            tt = item.type.tensor_type
            if not tt.HasField("shape"):
                return False
            for dim in tt.shape.dim:
                if dim.HasField("dim_param"):
                    return False
                if not dim.HasField("dim_value"):
                    return False
        return True
    except Exception:
        return False


_session_cache = {}

def get_session(raw: bytes):
    if ort is None:
        return True
    key = hashlib.sha256(raw).hexdigest()
    if key in _session_cache:
        return _session_cache[key]
    try:
        sess = ort.InferenceSession(raw, providers=["CPUExecutionProvider"])
    except Exception:
        sess = None
    _session_cache[key] = sess
    return sess

def run_binary_output(raw: bytes, x: np.ndarray):
    if ort is None:
        return None
    sess = get_session(raw)
    if sess is None:
        return None
    try:
        y = sess.run(["output"], {"input": x})[0]
    except Exception:
        return None
    return (y > 0.0).astype(np.float32)

def validate_task_model(raw: bytes, task_payload: dict):
    if ort is None:
        return True
    if get_session(raw) is None:
        return False
    subsets = ["train", "test", "arc-gen"] if use_full_validation else ["train", "test"]
    for subset in subsets:
        for ex in task_payload.get(subset, []):
            x = one_hot_grid(ex["input"])
            pred = run_binary_output(raw, x)
            if pred is None:
                return False
            tgt = one_hot_grid(ex["output"])
            if not np.array_equal(pred, tgt):
                return False
    return True


def estimate_cost(raw: bytes):
    if nu is not None:
        p = None
        try:
            with tempfile.NamedTemporaryFile(suffix=".onnx", delete=False) as tf:
                tf.write(raw)
                p = tf.name
            buf = io.StringIO()
            with contextlib.redirect_stdout(buf):
                macs, mem, params = nu.score_network(p)
            if None not in (macs, mem, params):
                return int(macs + mem + params)
        except Exception:
            pass
        finally:
            if p is not None:
                try:
                    os.remove(p)
                except Exception:
                    pass
    return len(raw)


def source_candidates(roots):
    zip_sources = []
    dir_sources = []
    for root in roots:
        if not root.exists():
            continue
        zips = sorted(root.rglob("submission.zip"))
        for z in zips:
            zip_sources.append(z)
        if not zips:
            dir_sources.append(root)
    return zip_sources, dir_sources


def make_transform_model(row_idx, col_idx, use_transpose=False, color_map=None):
    input_vi = onnx.helper.make_tensor_value_info("input", onnx.TensorProto.FLOAT, [1, 10, 30, 30])
    output_vi = onnx.helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT, [1, 10, 30, 30])
    nodes = []
    in_name = "input"
    if use_transpose:
        nodes.append(onnx.helper.make_node("Transpose", [in_name], ["t0"], perm=[0, 1, 3, 2]))
        in_name = "t0"
    r_tensor = onnx.helper.make_tensor("row_idx", onnx.TensorProto.INT64, [30], row_idx.astype(np.int64))
    c_tensor = onnx.helper.make_tensor("col_idx", onnx.TensorProto.INT64, [30], col_idx.astype(np.int64))
    nodes.append(onnx.helper.make_node("Constant", [], ["rows"], value=r_tensor))
    nodes.append(onnx.helper.make_node("Gather", [in_name, "rows"], ["g1"], axis=2))
    nodes.append(onnx.helper.make_node("Constant", [], ["cols"], value=c_tensor))
    nodes.append(onnx.helper.make_node("Gather", ["g1", "cols"], ["g2"], axis=3))
    if color_map is None:
        nodes.append(onnx.helper.make_node("Identity", ["g2"], ["output"]))
        inits = []
    else:
        w = np.zeros((10, 10, 1, 1), dtype=np.float32)
        for cin, cout in color_map.items():
            if 0 <= cin < 10 and 0 <= cout < 10:
                w[cout, cin, 0, 0] = 1.0
        inits = [onnx.helper.make_tensor("W", onnx.TensorProto.FLOAT, w.shape, w.flatten())]
        nodes.append(onnx.helper.make_node("Conv", ["g2", "W"], ["output"], kernel_shape=[1, 1]))
    graph = onnx.helper.make_graph(nodes, "m", [input_vi], [output_vi], inits)
    model = onnx.helper.make_model(graph, ir_version=10, opset_imports=[onnx.helper.make_opsetid("", 10)])
    try:
        onnx.checker.check_model(model, full_check=True)
        return model.SerializeToString()
    except Exception:
        return None


def apply_transform_np(x, transform_name):
    if transform_name == "id":
        return x
    if transform_name == "hflip":
        return x[:, :, ::-1]
    if transform_name == "vflip":
        return x[:, ::-1, :]
    if transform_name == "rot180":
        return x[:, ::-1, ::-1]
    if transform_name == "transpose":
        return np.transpose(x, (0, 2, 1))
    if transform_name == "rot90":
        return np.transpose(x, (0, 2, 1))[:, :, ::-1]
    if transform_name == "rot270":
        return np.transpose(x, (0, 2, 1))[:, ::-1, :]
    return x


def infer_color_map(task_payload, transform_name):
    mapping = {}
    subsets = ["train", "test"]
    for subset in subsets:
        for ex in task_payload.get(subset, []):
            x = one_hot_grid(ex["input"])[0]
            y = one_hot_grid(ex["output"])[0]
            xt = apply_transform_np(x, transform_name)
            for r in range(30):
                for c in range(30):
                    xi = xt[:, r, c]
                    yi = y[:, r, c]
                    if xi.sum() != 1.0 or yi.sum() != 1.0:
                        continue
                    cin = int(np.argmax(xi))
                    cout = int(np.argmax(yi))
                    prev = mapping.get(cin)
                    if prev is None:
                        mapping[cin] = cout
                    elif prev != cout:
                        return None
    for i in range(10):
        if i not in mapping:
            mapping[i] = i
    for subset in subsets:
        for ex in task_payload.get(subset, []):
            x = one_hot_grid(ex["input"])[0]
            y = one_hot_grid(ex["output"])[0]
            xt = apply_transform_np(x, transform_name)
            pred = np.zeros_like(y)
            for cin in range(10):
                pred[mapping[cin]] += xt[cin]
            pred = (pred > 0.0).astype(np.float32)
            if not np.array_equal(pred, y):
                return None
    return mapping


def synthesize_task_candidates(task_name, task_payload):
    row = np.arange(30, dtype=np.int64)
    rev = row[::-1].copy()
    specs = [
        ("id", row, row, False),
        ("hflip", row, rev, False),
        ("vflip", rev, row, False),
        ("rot180", rev, rev, False),
        ("transpose", row, row, True),
        ("rot90", row, rev, True),
        ("rot270", rev, row, True),
    ]
    out = []
    for tname, ridx, cidx, tflag in specs:
        raw_t = make_transform_model(ridx, cidx, tflag, None)
        if raw_t is not None:
            out.append((f"synth:{tname}", raw_t))
        cmap = infer_color_map(task_payload, tname)
        if cmap is not None:
            raw_tc = make_transform_model(ridx, cidx, tflag, cmap)
            if raw_tc is not None:
                out.append((f"synth:{tname}+cmap", raw_tc))
    uniq = {}
    for src, raw in out:
        h = hashlib.sha256(raw).hexdigest()
        if h not in uniq:
            uniq[h] = (src, raw)
    return list(uniq.values())


zip_sources, dir_sources = source_candidates(search_roots)

bank = defaultdict(list)
for z in zip_sources:
    try:
        models = load_zip_models(z)
        for k, v in models.items():
            bank[k].append((f"zip:{z.name}", v))
    except Exception:
        pass

for d in dir_sources:
    try:
        models = load_dir_models(d)
        for k, v in models.items():
            bank[k].append((f"dir:{d.name}", v))
    except Exception:
        pass

task_jsons = {}
comp_dir = Path("/kaggle/input/competitions/neurogolf-2026")
if comp_dir.exists():
    for jp in comp_dir.glob("task*.json"):
        tn = normalize_task_name(jp.stem)
        if tn is not None:
            task_jsons[tn] = json.loads(jp.read_text())

for task_name, payload in task_jsons.items():
    try:
        synth = synthesize_task_candidates(task_name, payload)
        for src, raw in synth:
            bank[task_name].append((src, raw))
    except Exception:
        pass

selected = {}
usage = Counter()
validation_failures = []
for task_name in sorted(bank.keys()):
    candidates = bank[task_name]
    candidates = sorted(candidates, key=lambda x: (source_rank(x[0]), len(x[1])))
    valid_pool = []
    for src, raw in candidates:
        if not is_processable_model(raw):
            continue
        valid_pool.append((src, raw))
        if len(valid_pool) >= MAX_VALID_CANDIDATES_PER_TASK:
            break

    chosen = None
    if task_name in task_jsons:
        payload = task_jsons[task_name]
        for src, raw in valid_pool:
            if not validate_task_model(raw, payload):
                continue
            chosen = (src, raw)
            break
        if chosen is None:
            validation_failures.append(task_name)

    if chosen is None and valid_pool:
        chosen = sorted(valid_pool, key=lambda t: len(t[1]))[0]
    if chosen is None and candidates:
        chosen = candidates[0]
    if chosen is not None:
        selected[task_name] = chosen[1]
        usage[chosen[0]] += 1

if selected:
    manifest = manifest_from_items(list(selected.items()))
    if manifest == EXPECTED_MANIFEST_SHA256:
        print("Matched expected 6335.19 manifest")
    else:
        print("Manifest differs from expected 6335.19")
        print(manifest)

with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in sorted(selected.keys()):
        zf.writestr(name, selected[name])

zip_sha = hashlib.sha256(OUTPUT_ZIP.read_bytes()).hexdigest()
print(f"Wrote {OUTPUT_ZIP}")
print(f"ONNX files: {len(selected)}")
print(f"zip sha256: {zip_sha}")
print(f"sources used: {dict(usage)}")
if validation_failures:
    print(f"Validation misses: {len(validation_failures)}")
    print(validation_failures[:20])
if len(selected) != EXPECTED_FILE_COUNT:
    print(f"Warning: expected {EXPECTED_FILE_COUNT}, found {len(selected)}")


## Attribution

This release builds on the public NeuroGolf notebook ecosystem and my previous public artifact. Useful public references include:

- `afr1ste/neurogolf-6285-95-public-score-open-solution`
- `artemnazemtsev/neuro-golf-6312`
- `artemnazemtsev/neurogolf-acking-multiple-tasks`
- `artemnazemtsev/neurogolf-blending-is-all-you-need`
- `magmacot/neurogolf-new-blending`
- `jonathanchan/ngc26-constraint-smart-logic-mix-blending`

The point of listing these is practical traceability: public-score artifacts are easier to learn from when the lineage is explicit.
